In [4]:
import pandas as pd
from datetime import timedelta

df = pd.read_csv('cleaned_retail_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])  # re-convert, since CSVs don't keep this

In [5]:
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

def segment(row):
    if row['RFM_Score'] in ['444','434','443']:
        return 'Champions'
    elif row['R_score'] in [3,4] and row['F_score'] in [3,4]:
        return 'Loyal Customers'
    elif row['R_score'] in [1,2] and row['F_score'] in [3,4]:
        return 'At Risk'
    elif row['R_score'] == 1 and row['F_score'] == 1:
        return 'Lost'
    else:
        return 'Potential Loyalist'

rfm['Segment'] = rfm.apply(segment, axis=1)
rfm.to_csv('rfm_segments.csv')
print(rfm['Segment'].value_counts())

Segment
Potential Loyalist    2167
Loyal Customers       1220
At Risk                880
Champions              839
Lost                   772
Name: count, dtype: int64


In [6]:
rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

Segment
Champions             9883515.995
Loyal Customers       3446018.604
At Risk               2165027.544
Potential Loyalist    1626686.003
Lost                   253556.122
Name: Monetary, dtype: float64

In [7]:
revenue_by_segment = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
revenue_by_segment_pct = (revenue_by_segment / revenue_by_segment.sum() * 100).round(1)
print(revenue_by_segment_pct)

Segment
Champions             56.9
Loyal Customers       19.8
At Risk               12.5
Potential Loyalist     9.4
Lost                   1.5
Name: Monetary, dtype: float64


In [8]:
rfm.to_csv('rfm_segments.csv')
print("Saved.")

Saved.
